# 🥇 Gold — Hourly demand

**Purpose.** Aggregate trip counts by hour-of-day × day-of-week — the canonical taxi-demand heatmap.

**Source:** `workspace.silver.yellow_taxi`
**Output:** `workspace.gold.hourly_demand` (24 × 7 = 168 rows)

**Why this aggregate exists.** A dashboard showing demand patterns shouldn't run a heavy GROUP BY across 5M+ rows on every refresh. Pre-aggregating to 168 rows makes the heatmap instant.

In [0]:
from pyspark.sql.functions import col, hour, dayofweek, date_format, count, avg, round as spark_round

SOURCE_TABLE = "workspace.silver.yellow_taxi"
FULL_TABLE_NAME = "workspace.gold.hourly_demand"

silver_df = spark.table(SOURCE_TABLE)

# dayofweek() returns 1=Sunday through 7=Saturday in Spark.
# We also store a human-readable day name for dashboard labels.
hourly_df = (
    silver_df
    .withColumn("hour_of_day", hour("tpep_pickup_datetime"))
    .withColumn("day_of_week_num", dayofweek("tpep_pickup_datetime"))
    .withColumn("day_of_week", date_format("tpep_pickup_datetime", "EEEE"))  # "Monday"
    .groupBy("day_of_week_num", "day_of_week", "hour_of_day")
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
    )
    .orderBy("day_of_week_num", "hour_of_day")
)

row_count = hourly_df.count()
print(f"Hourly demand aggregate: {row_count} rows (expected 168)")

# Write to Delta.
(
    hourly_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)
print(f"✓ Wrote {row_count} rows to {FULL_TABLE_NAME}")

In [0]:
result = spark.table(FULL_TABLE_NAME)

print("Top 5 busiest hour-of-week slots:")
display(result.orderBy(col("trip_count").desc()).limit(5))

print("\nQuietest 5 hour-of-week slots:")
display(result.orderBy(col("trip_count").asc()).limit(5))